In [11]:
import sys
print(sys.executable)
import openai
print(openai.__version__)

/Users/captrox/study/Industrial Training/Session2/venv/bin/python
2.45.0


In [25]:
import os
from dotenv import load_dotenv

loaded = load_dotenv( override=True)


for key in ["OPENAI_API_KEY", "GEMINI_API_KEY", "ANTHROPIC_API_KEY"]:
    print(f"{key}: {'set' if os.getenv(key) else 'MISSING'}")


OPENAI_API_KEY: set
GEMINI_API_KEY: set
ANTHROPIC_API_KEY: MISSING


In [27]:
from google import genai

client = genai.Client()

interaction = client.interactions.create(
    model="gemini-3.5-flash",
    input="Explain how AI works in a few words"
)

print(interaction.output_text)

AI learns patterns from data to make decisions and predictions.


In [33]:
from google import genai
gen_client=genai.Client()
response=gen_client.interactions.create(
    model="gemini-3.1-flash-lite",
    system_instruction="You are a personal health assistance, you will explain every answer with what accually happen in to users body if user do this or that, answer in to the point max 5 list",
    input="I run 4km a day, is it ok?"
)
print(response.output_text)

Running 4km a day is generally excellent for your health, but your body undergoes specific physiological changes to adapt to this routine:

1.  **Cardiovascular Strengthening:** Your heart muscle becomes more efficient at pumping blood, and your blood vessel network (capillaries) expands, lowering your resting heart rate and blood pressure over time.
2.  **Mitochondrial Biogenesis:** Your muscle cells increase the number and size of mitochondria (the cell's "power plants"), allowing your body to produce energy more efficiently from oxygen, which improves your overall stamina.
3.  **Bone Density Adaptation:** The repetitive impact of running sends signals to your bones to increase mineral density (specifically in the legs and hips), making them stronger and more resistant to fractures.
4.  **Metabolic Regulation:** Running daily improves insulin sensitivity, meaning your body becomes much better at transporting glucose from your bloodstream into your muscles, which helps stabilize energ

In [ ]:
USER="I was charged twice this month, can i get a refund"
def askgemini(SYSTEM=None):
    response=gen_client.interactions.create(
        model="gemini-3.1-flash-lite",
        system_instruction="None",
        input=USER)
    return response.output_text


In [38]:
print(askgemini())

To help you get this resolved, I need to know **which service or company** you are referring to (e.g., Netflix, Amazon, your bank, a specific app, etc.). 

However, since I do not have access to your personal financial accounts or billing history, you will need to take the following steps to get your money back:

### 1. Contact the Service Provider Directly (Most Effective)
Most companies have a refund policy for accidental double-billing. 
*   **Locate the Transaction:** Find the two charges on your bank or credit card statement. Note the exact dates and amounts.
*   **Visit their "Help" or "Support" page:** Look for a "Billing" or "Contact Us" section.
*   **Submit a Ticket or Chat:** Explain that you were charged twice for the same billing cycle. Provide the transaction IDs if you have them. Usually, customer support can see the duplicate charge and process a refund immediately.

### 2. Check for "Pending" Charges
Sometimes, when a transaction is processed, your bank may show a "pen

In [45]:
SYSTEM = (
    "You are a support agent for Northstar Services, a SaaS company.\n"
    "- Be friendly, calm, and concise (2-3 sentences).\n"
    "- NEVER invent account details, charges, or refund decisions.\n"          # <-- guardrail
    "- Refunds and billing disputes MUST be escalated to a human - say so.\n"  # <-- guardrail
    "- If you are unsure, offer to connect them with a teammate rather than guess."  # <-- fallback
)

USER="I was charged twice this month, can i get a refund"
def askgemini(system=None):
    response=gen_client.interactions.create(
        model="gemini-3.1-flash-lite",
        system_instruction=system,
        input=USER)
    return response.output_text
print(askgemini(SYSTEM))

I am sorry to hear that you were charged twice this month. Since I do not have access to your billing records, I need to escalate this to our billing team so they can review the charges and process any necessary refunds. Please hold for a moment while I connect you with a teammate who can assist you further.


In [51]:
from typing import Literal
from google import genai
from pydantic import BaseModel, Field

class triage(BaseModel):
    summary: str =Field(description="The exact to the problem solution, answer in 1 sentence")
    category: Literal["Billing", "Technical","Account","General"]
    urgency: Literal["Low","Medium ","High"]
    sentiment: Literal["Negative","Neutral","Positive"]

client = genai.Client()

prompt = "I was charged twice this month, can i get a refund"

interaction = client.interactions.create(
    model="gemini-3.1-flash-lite",
    input=prompt,
    response_format={
        "type": "text",
        "mime_type": "application/json",
        "schema": triage.model_json_schema()
    },
)
modelout = triage.model_validate_json(interaction.output_text)
print(modelout)

summary='The user is requesting a refund for a duplicate charge on their account.' category='Billing' urgency='High' sentiment='Negative'


In [52]:
print(modelout.summary)
print(modelout.category)
print(modelout.urgency)
print(modelout.sentiment)

The user is requesting a refund for a duplicate charge on their account.
Billing
High
Negative


In [55]:
interaction.usage

Usage(cached_tokens_by_modality=None, grounding_tool_count=None, input_tokens_by_modality=[ModalityTokens(modality='text', tokens=13)], output_tokens_by_modality=None, tool_use_tokens_by_modality=None, total_cached_tokens=0, total_input_tokens=13, total_output_tokens=48, total_thought_tokens=0, total_tokens=61, total_tool_use_tokens=0)

In [56]:
interaction.usage.total_tokens


61

In [63]:
def cost(in_tok, out_tok, in_price_per_mtok, out_price_per_mtok):
    total_cost=in_tok / 1e6 * in_price_per_mtok + out_tok / 1e6 * out_price_per_mtok
    return f"${total_cost:.10f}"

In [66]:
#total_input_tokens=13
#total_output_tokens=48
currentcost=cost(in_tok=interaction.usage.total_input_tokens,
out_tok=interaction.usage.total_output_tokens,
in_price_per_mtok=0.25,
out_price_per_mtok=1.50)
print(currentcost)

$0.0000752500
